<a href="https://colab.research.google.com/github/jinw00ch01/Dacon_RobotWorldModel_ActionVideo/blob/main/colab/train_cosmos_ac_v5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# v5: 학습 클립 5.4배 + 잠재 IDM 보조 손실 (Colab A100/H100)

Ultra 가 만든 잠재 캐시 두 개(`latents_240x320` 2.35만 클립 + `latents_240x320_extra` 10.3만 클립)를 Google Drive `MyDrive/dacon_wm` 에서 읽고, 체크포인트를 같은 곳에 저장한다.
끊기면 **모두 실행**을 다시 누르면 잠재 IDM 은 건너뛰고 마지막 체크포인트부터 이어서 학습한다.

바뀐 점 (제출 3·5 의 v2_long 16,000 스텝에서 이어서):
- 클립 12.7만 개를 약 1 에폭 (배치 8 × 16,000 스텝). 지금까지는 2.35만 개를 2.7 에폭 반복해 16k 이후 과적합됐다.
- 잠재 IDM 보조 손실: 잠재에서 상대 행동을 읽는 작은 모델을 먼저 학습해 고정하고, 스텝의 30% 에서 생성기가 예측한 깨끗한 잠재가 명령한 움직임을 담도록 L1 0.3 배를 더한다 (낮은 잡음 0.05~0.45 에서).
- 잡음 수준 샘플링을 추론 샘플러와 같은 shift 5 로 맞춘다. 움직임이 많은 클립을 더 자주 본다 (motion weight 1).
- 학습률은 일정(LoRA 5e-5, 임베더 1.5e-4), 500 스텝마다 adapter 를 남겨 마지막 체크포인트 평균에 쓴다.
- 숫자는 모두 학습 데이터 또는 공개된 2등 해법 설정에서 왔다. 평가 데이터는 쓰지 않는다 (규칙 3).

준비 (사람): Drive 의 `dacon_wm` 에 `latents_240x320_extra` 가 다 올라가 있어야 한다. 런타임 유형은 GPU (A100 또는 H100).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
SHARED = '/content/drive/MyDrive/dacon_wm'
assert os.path.isdir(SHARED), 'Drive 에 dacon_wm 폴더가 없습니다'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 코드: 공개 저장소의 main (실행한 커밋은 아래에 출력되고 체크포인트 config 에도 남는다)
!rm -rf /content/repo && git clone -q https://github.com/jinw00ch01/Dacon_RobotWorldModel_ActionVideo /content/repo
!cd /content/repo && git log --oneline -1
!pip -q install diffusers==0.40.0 peft==0.21.2 transformers==5.18.0 accelerate safetensors einops av opencv-python-headless pandas pyarrow
# Colab 에 깔린 torchao 0.10 은 peft 0.21 이 LoRA 를 만들 때 거부한다. 여기서는 쓰지 않으므로 지운다.
!pip -q uninstall -y torchao

In [ ]:
# 잠재 캐시 두 개를 로컬 디스크로 복사 (Drive 에서 직접 읽으면 느리다)
!mkdir -p /content/latents /content/latents_extra
!rsync -a --info=progress2 "$SHARED/latents_240x320/" /content/latents/
!rsync -a --info=progress2 "$SHARED/latents_240x320_extra/" /content/latents_extra/
os.environ['WM_SHARED'] = SHARED
os.environ['WM_ACTION_STATS'] = f'{SHARED}/so100_action_statistics.json'
import glob
n_base, n_extra = len(glob.glob('/content/latents/*.pt')), len(glob.glob('/content/latents_extra/*.pt'))
print('shards', n_base, n_extra)
assert n_base == 116, f'latents_240x320 가 {n_base}/116 개입니다'
assert os.path.exists(f'{SHARED}/latents_240x320_extra/manifest_train.json'), 'latents_240x320_extra 업로드가 끝나지 않았습니다 (manifest_train.json 없음)'

In [ ]:
# 잠재 IDM: 잠재 (16, 5, 30, 40) -> 클립의 상대 행동 16x6. 학습 캐시만 쓰고 2% 는 검증으로 남긴다 (A100 약 15~25분)
LIDM = f'{SHARED}/cosmos_ac/latent_idm/latent_idm_v1.pt'
if not os.path.exists(LIDM):
    !cd /content/repo && python -m wmgen.latent_idm --latents /content/latents /content/latents_extra --out "$LIDM" --steps 25000 --batch 64
else:
    print('잠재 IDM 이미 있음:', LIDM)

In [ ]:
# v5: v2_long 16,000 스텝에서 이어서 32,000 스텝까지 (배치 8). 끊기면 이 셀만 다시 실행해도 이어진다.
import shutil, torch
SRC = f'{SHARED}/cosmos_ac/runs/v2_long'
RUN = f'{SHARED}/cosmos_ac/runs/v5'
os.makedirs(RUN, exist_ok=True)
if not os.path.exists(f'{RUN}/adapter_last.pt'):
    for f in ('adapter_last.pt', 'optimizer_last.pt'):
        shutil.copy(f'{SRC}/{f}', f'{RUN}/{f}')
    print('v2_long 16,000 스텝에서 시작')
BATCH = 8
# 80GB GPU (A100 80GB, H100) 면 gradient checkpointing 을 끄고 더 빨리 돈다. 40GB 는 켠 채로 둔다.
GC = '--no-grad-ckpt' if torch.cuda.get_device_properties(0).total_memory > 70 * 2**30 else ''
ABS = ''  # 절대 관절 채널 설정 (진단 D1 결과로 정함)
!cd /content/repo && python -m wmgen.train_cosmos_ac --latents /content/latents /content/latents_extra --out "$RUN" --batch $BATCH --steps 32000 --save-every 500 --keep-every 500 --lr 5e-5 --embedder-lr 1.5e-4 --lr-schedule constant --motion-weight 1.0 --sigma-shift 5 --aux-idm "$LIDM" --aux-prob 0.3 --aux-weight 0.3 $GC $ABS